In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("source_table", "spotifyproject.bronze.tracks_features")
dbutils.widgets.text("target_table", "spotifyproject.silver.tracks_features")

SOURCE_TABLE = dbutils.widgets.get("source_table")
TARGET_TABLE = dbutils.widgets.get("target_table")

In [0]:
bronze = spark.table(SOURCE_TABLE)

diag = (
    bronze
    .withColumn("year_int", col("year").cast("int"))
    .withColumn("date_year", year(try_to_date(col("release_date"), "yyyy-MM-dd")))
)

print("year unparseable as int:")
diag.filter(col("year_int").isNull()).select("id", "name", "year", "release_date").show()

print("year <1900 or >2026:")
diag.filter((col("year_int") < 1900) | (col("year_int") > 2026)) \
    .select("id", "name", "year", "release_date").show()

print("year not equal with release_date:")
diag.filter(col("year_int") != col("date_year")) \
    .select("id", "name", "year", "release_date").show()

print("release_date unparseable:")
diag.filter(try_to_date(col("release_date"), "yyyy-MM-dd").isNull()) \
    .select("id", "name", "year", "release_date").show()

In [0]:
parsed_release_date = coalesce(
    try_to_date(col("release_date"), "yyyy-MM-dd"),
    try_to_date(concat_ws("-", col("release_date"), lit("01")), "yyyy-MM-dd"),
    try_to_date(concat_ws("-", col("release_date"), lit("01"), lit("01")), "yyyy-MM-dd"),
)

CURRENT_YEAR = 2026

silver = (
    bronze
    .withColumn("release_date_parsed", parsed_release_date)
    .withColumn("year_raw", col("year").cast(IntegerType()))
    .withColumn("year_from_date", year(col("release_date_parsed")))

    .withColumn(
        "year",
        when(
            col("year_raw").isNull()
            | (col("year_raw") < 1900)
            | (col("year_raw") > CURRENT_YEAR),
            col("year_from_date"),
        ).otherwise(col("year_raw")),
    )
    .withColumn(
        "year_was_repaired",
        (col("year_raw").isNull())
        | (col("year_raw") < 1900)
        | (col("year_raw") > CURRENT_YEAR),
    )
    .withColumn("year_date", try_to_date(col("year").cast("string"), "yyyy"))

    .withColumn("name", trim(col("name")))
    .withColumn("album", trim(col("album")))
    .withColumn(
        "artists_array",
        split(regexp_replace(col("artists"), r"^\[['\"]|['\"]\]$", ""), r"['\"]\s*,\s*['\"]"),
    )
    .withColumn(
        "artist_ids_array",
        split(regexp_replace(col("artist_ids"), r"^\[['\"]|['\"]\]$", ""), r"['\"]\s*,\s*['\"]"),
    )
    .withColumn("artists", concat_ws(", ", col("artists_array")))
    .withColumn("artist_ids", concat_ws(", ", col("artist_ids_array")))
    .withColumn("artist_count", size(col("artists_array")))

    .withColumn("explicit", col("explicit").cast(BooleanType()))
    .withColumn("danceability",     round(col("danceability").cast(DoubleType()), 2))
    .withColumn("energy",           round(col("energy").cast(DoubleType()), 2))
    .withColumn("loudness",         round(col("loudness").cast(DoubleType()), 2))
    .withColumn("speechiness",      round(col("speechiness").cast(DoubleType()), 2))
    .withColumn("acousticness",     round(col("acousticness").cast(DoubleType()), 2))
    .withColumn("instrumentalness", round(col("instrumentalness").cast(DoubleType()), 2))
    .withColumn("liveness",         round(col("liveness").cast(DoubleType()), 2))
    .withColumn("valence",          round(col("valence").cast(DoubleType()), 2))
    .withColumn("tempo",            col("tempo").cast(DoubleType()).cast(IntegerType()))
    .withColumn("key",              col("key").cast(IntegerType()))
    .withColumn("mode",             col("mode").cast(IntegerType()))
    .withColumn("duration_ms",      col("duration_ms").cast(IntegerType()))
    .withColumn("duration_s",       round(col("duration_ms") / 1000).cast(IntegerType()))
    .withColumn("_transformed_at",  current_timestamp())
)

In [0]:
silver_final = silver.select(
    "id", "name", "album", "artists", "artist_ids", "artists_array", "artist_count",
    "explicit", "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness", "liveness", "valence",
    "tempo", "duration_ms", "duration_s",
    "year", "year_date", "release_date_parsed", "year_was_repaired",
    "_source_file", "_ingested_at", "_transformed_at",
).filter(col('year') > 1900)

In [0]:
(
    silver_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)